# ViFinQA — LLM chọn dòng (Qwen3-8B)

Chạy trên Colab T4. Đọc `batch_*.jsonl`, ghi `decisions.jsonl`.

**Ràng buộc:** model phải < 14B tham số. Qwen3-8B = 8.2B ✓.
KHÔNG đổi sang Qwen3-14B (~14.7B) — vi phạm thể lệ.

**Bất biến:** model chỉ trả về `chosen_index`. Không bao giờ trả giá trị đáp án.

In [ ]:
# vllm==0.6.3 predates Qwen3 support entirely (no Qwen3ForCausalLM
# architecture registered) -- LLM(model="Qwen/Qwen3-8B") fails to load.
# Bump to a vLLM release new enough to have Qwen3 support.
!pip -q install "vllm>=0.8.5" "huggingface_hub>=0.24"

In [ ]:
from google.colab import files
import pathlib

pathlib.Path("batches").mkdir(exist_ok=True)
print("Chọn toàn bộ batch_*.jsonl đã sinh bằng `submission row-batches`:")
uploaded = files.upload()
for name in uploaded:
    pathlib.Path("batches", name).write_bytes(uploaded[name])
print("đã nhận:", sorted(p.name for p in pathlib.Path("batches").glob("*.jsonl")))

In [ ]:
from vllm import LLM, SamplingParams

# Design doc (2026-08-22 Sec 5) specifies Qwen3-8B q4 (quantized), not
# full fp16. fp16 Qwen3-8B is ~16.4GB of weights alone -- already over a
# T4's 15GB VRAM budget before any KV cache, so dtype="half" at
# gpu_memory_utilization=0.90 OOMs before the first token.
# Loading a pre-quantized AWQ checkpoint (~5-6GB int4 weights) leaves
# headroom for the KV cache within budget. If this exact repo is
# unavailable, substitute any Qwen3-8B AWQ/GPTQ mirror and keep
# quantization="awq" (or "gptq") to match.
MODEL = "Qwen/Qwen3-8B-AWQ"  # 8.2B params < 14B. KHONG doi sang 14B.
llm = LLM(
    model=MODEL,
    quantization="awq",
    dtype="float16",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
)
# max_tokens bumped from 16 -> 64 (see cell 4): Qwen3's chat template
# defaults to <think>...</think> reasoning content, which would consume
# the whole budget before any JSON is emitted.
sampling = SamplingParams(temperature=0.0, max_tokens=64)

In [ ]:
# Raw-completion API (not a chat template) is used here, so Qwen3's
# enable_thinking=False chat-template kwarg is not reachable. The
# instruction above (no <think>, no giai thich) plus the higher
# max_tokens budget in cell 3 (64 vs. the old 16) gives parse() a
# realistic chance to find the JSON even if the model still emits a
# short preamble before it.
import json, pathlib, re

PROMPT = """Bạn là trợ lý phân tích báo cáo tài chính.
Không giải thích, không suy luận, không dùng <think>.

Câu hỏi: {question}

Các dòng ứng viên:
{candidates}

Chọn ĐÚNG MỘT dòng trả lời câu hỏi. Chỉ trả về JSON, không kèm gì khác: {{"chosen_index": <số>}}"""


def render(payload):
    lines = []
    for c in payload["candidates"]:
        parts = [f'[{c["index"]}] {c["row_label"]}']
        if c.get("row_group_context"):
            parts.append(f'(mục: {c["row_group_context"]})')
        if c.get("table_title"):
            parts.append(f'(bảng: {c["table_title"]})')
        if c.get("periods"):
            parts.append(f'(kỳ: {", ".join(c["periods"])})')
        lines.append(" ".join(parts))
    return PROMPT.format(question=payload["question"], candidates="\n".join(lines))


def parse(text, limit):
    match = re.search(r'"chosen_index"\s*:\s*(-?\d+)', text)
    if match is None:
        match = re.search(r"-?\d+", text)
    if match is None:
        return 0
    value = int(match.group(1) if match.lastindex else match.group(0))
    return value if 0 <= value < limit else 0


out = pathlib.Path("decisions.jsonl")
done = set()
if out.exists():  # chạy lại sau timeout chỉ tốn phần còn thiếu
    done = {json.loads(l)["question_id"] for l in out.read_text("utf-8").splitlines() if l.strip()}
    print("đã có sẵn:", len(done))

with out.open("a", encoding="utf-8") as sink:
    for batch in sorted(pathlib.Path("batches").glob("batch_*.jsonl")):
        payloads = [json.loads(l) for l in batch.read_text("utf-8").splitlines() if l.strip()]
        payloads = [p for p in payloads if p["question_id"] not in done and p["candidates"]]
        if not payloads:
            continue
        outputs = llm.generate([render(p) for p in payloads], sampling)
        for payload, output in zip(payloads, outputs):
            index = parse(output.outputs[0].text, len(payload["candidates"]))
            sink.write(json.dumps({"question_id": payload["question_id"],
                                   "chosen_index": index}) + "\n")
        sink.flush()
        print(batch.name, "xong", len(payloads), "câu")

In [ ]:
from google.colab import files
files.download("decisions.jsonl")